# Naive Bayes

Datasets:
- https://www.kaggle.com/datasets/ashfakyeafi/spam-email-classification

In [20]:
import sys
sys.path.append("..")

In [21]:
from impl.metrics import train_test_split

In [22]:
import pandas as pd

df = pd.read_csv("../datasets/email.csv")
X, y = df["Message"].to_numpy(), df["Category"].map({"spam": 1, "ham": 0}).to_numpy()

In [23]:
X_train, y_train, X_test, y_test = train_test_split(X, y, 0.65)

In [ ]:
import re
from typing import Literal

import numpy as np
from numpy.typing import NDArray


def create_corpus(emails: NDArray[np.str_]):
    words = set()
    for email in emails:
        for word in re.findall(r"\w+", email):
            words.add(word)

    return tuple(words)

def one_hot_encode(corpus: tuple, email: str) -> NDArray:
    ohe = np.array([int(word in email) for word in corpus])
    return ohe

class NaiveBayesClassifier:
    def __init__(self):
        self.phi_0 = {}
        self.phi_1 = {}
        self.phi_y: float | None = None

        self._corpus: tuple | None = None

    def fit(self, X: NDArray[np.str_], y: NDArray):
        corpus = create_corpus(X)

        self._corpus = corpus

        self.phi_y = np.sum(y) / len(y)
        c1, c0 = sum(int(y_i == 1) for y_i in y), sum(int(y_i == 0) for y_i in y)

        for j, word in enumerate(corpus):
            s1 = 0
            s0 = 0

            for x_i, y_i in zip(X, y):
                if word in x_i and y_i == 1:
                    s1 += 1
                elif word in x_i and y_i == 0:
                    s0 += 0

            phi_j_1 = s1 / c1
            phi_j_0 = s0 / c0

            self.phi_0[j] = phi_j_0
            self.phi_1[j] = phi_j_1

    def _p_x(self, X: NDArray):
        assert self.phi_y
        p1 = 1
        p0 = 1
        for j, val in enumerate(X):
            if val == 1:
                p1 *= self.phi_1[j]
                p0 *= self.phi_0[j]
            else:
                assert val == 0
                p1 *= (1 - self.phi_1[j])
                p0 *= (1 - self.phi_0[j])

        return (p1 * self.phi_y) + (p0 * (1 - self.phi_y))

    def _p_of_y_given_x(self, X: NDArray, y: Literal[0, 1]):
        assert self.phi_y

        main_term = 1
        if y == 1:
            for j, val in enumerate(X):
                p = self.phi_1[j] if val == 1 else (1 - self.phi_1[j])
                main_term *= p
        else:
            assert y == 0
            for j, val in enumerate(X):
                p = self.phi_0[j] if val == 1 else (1 - self.phi_0[j])
                main_term *= p

        sub_term = self.phi_y if y == 1 else (1 - self.phi_y)

        return (main_term * sub_term) / self._p_x(X)

    def predict(self, X: str) -> Literal[0, 1]:
        assert self.phi_0 and self.phi_1 and self.phi_y and self._corpus

        X_ohe = one_hot_encode(self._corpus, X) # type: ignore

        p1 = self._p_of_y_given_x(X_ohe, 1)
        p0 = self._p_of_y_given_x(X_ohe, 0)

        if p1 >= p0:
            return 1
        else:
            return 0

In [25]:
model = NaiveBayesClassifier()
model.fit(X_train, y_train)

In [26]:
correct = 0
for idx, X_i in enumerate(X_test):
    prediction = model.predict(X_i)

    if prediction == y_test[idx]:
        correct += 1

print(f"{correct}/{len(X_test)} correct {(correct/len(X_test)) * 100}% accuracy")

/var/folders/2x/jhz5n62j1ldb7x0_f_q2424c0000gn/T/ipykernel_10697/385315106.py:83: RuntimeWarning: invalid value encountered in scalar divide
  return (main_term * sub_term) / self._p_x(X)


1724/1950 correct 88.41025641025641% accuracy
